In [1]:
!pip install ir_datasets

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/866.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/866.1 kB ? eta -:--:--
   ------------ --------------------------- 262.1/866.1 kB ? eta -:--:--
   ------------ --------------------------- 262.1/866.1 kB ? eta -:--:--
   ------------ --------------------------- 262.1/86

In [2]:
from services.dataset_service import *
from services.preprocessing_service import *
from services.indexing_service import *

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
dataset = load_dataset()

print_dataset_info(
    dataset
)

DATASET
Name: CORD-19 TREC-COVID
Docs: 192509
Queries: 50
Qrels: 69318


In [ ]:
doc = next(dataset.docs_iter())

text = extract_text(doc)

print("ORIGINAL:")
print(text[:400])

print()

print("PROCESSED:")
print(preprocess(text[:400]))

[INFO] [starting] building docstore
[INFO] If you have a local copy of https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-07-16/metadata.csv, you can symlink it here to avoid downloading it again: C:\Users\Administrator\.ir_datasets\downloads\80d664e496b8b7e50a39c6f6bb92e0ef
[INFO] download error: HTTPSConnectionPool(host='ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com', port=443): Max retries exceeded with url: /2020-07-16/metadata.csv (Caused by NameResolutionError("HTTPSConnection(host='ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com', port=443): Failed to resolve 'ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com' ([Errno 11001] getaddrinfo failed)")). Retrying from start.
[INFO] download error: HTTPSConnectionPool(host='ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com', port=443): Max retries exceeded with url: /2020-07-16/metadata.csv (Caused by NameResolutionError("HTTPSConnection(host='ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws

ConnectionError: HTTPSConnectionPool(host='ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com', port=443): Max retries exceeded with url: /2020-07-16/metadata.csv (Caused by NameResolutionError("HTTPSConnection(host='ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com', port=443): Failed to resolve 'ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com' ([Errno 11002] getaddrinfo failed)"))

In [ ]:
d1_docs, d1_queries = preprocess_dataset(dataset, "cord19.pkl")



In [ ]:
d1_index, d1_lengths = build_inverted_index(
    dataset,
    preprocess,
    "cord19_index.pkl"
)



In [ ]:
print(dataset)
print(type(next(dataset.docs_iter())))

In [1]:
import ir_datasets
from collections import Counter
from services.preprocessing_service import preprocess
from services.indexing_service import build_inverted_index
from services.retrieval_service import RetrievalService

print("1. جاري تحميل مجموعة بيانات تجريبية صغيرة...")
mini_dataset = ir_datasets.load("cranfield")
total_docs = mini_dataset.docs_count()
print(f"تم تحميل البيانات بنجاح! عدد الوثائق: {total_docs}")

print("\n2. جاري معالجة البيانات وبناء الفهرس المقلوب...")

index, doc_lengths = build_inverted_index(mini_dataset, preprocess, "cranfield_index.pkl")
print("تم بناء الفهرس المقلوب بنجاح!")

print("\n3. تهيئة خدمة البحث والاسترجاع (الطلب الثاني)...")
retrieval = RetrievalService(index, doc_lengths, total_docs)


user_query = "aerodynamic heating structural behavior"
query_tokens = preprocess(user_query)
print(f"الاستعلام بعد المعالجة: {query_tokens}")


print("\n--- نتائج البحث باستخدام VSM (TF-IDF) ---")
results_tfidf = retrieval.search_tfidf(query_tokens, top_k=5)
for rank, (doc_id, score) in enumerate(results_tfidf, 1):
    print(f"المرتبة {rank}: معرف الوثيقة = {doc_id} | النتيجة (تشابه الجيوب) = {score:.4f}")

print("\n--- نتائج البحث باستخدام BM25 ---")
results_bm25 = retrieval.search_bm25(query_tokens, top_k=5)
for rank, (doc_id, score) in enumerate(results_bm25, 1):
    print(f"المرتبة {rank}: معرف الوثيقة = {doc_id} | النتيجة = {score:.4f}")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


1. جاري تحميل مجموعة بيانات تجريبية صغيرة...
تم تحميل البيانات بنجاح! عدد الوثائق: 1400

2. جاري معالجة البيانات وبناء الفهرس المقلوب...
Building index: cranfield_index.pkl
[CACHE SAVED] cache\cranfield_index.pkl
Saved index cache: cranfield_index.pkl
تم بناء الفهرس المقلوب بنجاح!

3. تهيئة خدمة البحث والاسترجاع (الطلب الثاني)...
الاستعلام بعد المعالجة: ['aerodynam', 'heat', 'structur', 'behavior']

--- نتائج البحث باستخدام VSM (TF-IDF) ---
المرتبة 1: معرف الوثيقة = 51 | النتيجة (تشابه الجيوب) = 0.7670
المرتبة 2: معرف الوثيقة = 142 | النتيجة (تشابه الجيوب) = 0.6845
المرتبة 3: معرف الوثيقة = 860 | النتيجة (تشابه الجيوب) = 0.6150
المرتبة 4: معرف الوثيقة = 1361 | النتيجة (تشابه الجيوب) = 0.5890
المرتبة 5: معرف الوثيقة = 724 | النتيجة (تشابه الجيوب) = 0.5577

--- نتائج البحث باستخدام BM25 ---
المرتبة 1: معرف الوثيقة = 51 | النتيجة = 8.5908
المرتبة 2: معرف الوثيقة = 142 | النتيجة = 7.4382
المرتبة 3: معرف الوثيقة = 860 | النتيجة = 6.7970
المرتبة 4: معرف الوثيقة = 1361 | النتيجة = 6.3994
المر

In [3]:
!pip install sentence-transformers

  Using cached sentence_transformers-5.5.1-py3-none-any.whl.metadata (18 kB)
  Using cached transformers-5.12.0-py3-none-any.whl.metadata (33 kB)
  Using cached huggingface_hub-1.19.0-py3-none-any.whl.metadata (14 kB)
  Using cached torch-2.12.0-cp312-cp312-win_amd64.whl.metadata (31 kB)
  Using cached scikit_learn-1.9.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.26.7-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached filelock-3.29.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.5.1-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached 

In [1]:
import ir_datasets
from services.preprocessing_service import preprocess
from services.indexing_service import build_inverted_index
from services.retrieval_service import RetrievalService

print("1. جاري تحميل مجموعة البيانات التجريبية...")

mini_dataset = ir_datasets.load("cranfield")
total_docs = mini_dataset.docs_count()

print("2. جاري بناء الفهرس المقلوب الإحصائي...")
index, doc_lengths = build_inverted_index(mini_dataset, preprocess, "cranfield_index.pkl")

print("3. تهيئة خدمة البحث الشاملة (الطلب الثاني)...")
retrieval = RetrievalService(index, doc_lengths, total_docs)

print("\n4. جاري توليد الـ Embeddings للوثائق باستخدام BERT...")

retrieval.compute_documents_embeddings(mini_dataset)


user_query = "aerodynamic heating structural behavior"
query_tokens = preprocess(user_query)

print(f"\nالاستعلام النصي الأصلي: '{user_query}'")
print(f"الاستعلام بعد المعالجة (Tokens): {query_tokens}\n")


print("="*50)
print("--- 1. نتائج تمثيل VSM (TF-IDF) ---")
print(retrieval.search_tfidf(query_tokens, top_k=3))

print("\n--- 2. نتائج تمثيل BM25 ---")
print(retrieval.search_bm25(query_tokens, top_k=3))

print("\n--- 3. نتائج تمثيل الـ Embeddings (BERT) ---")
print(retrieval.search_embeddings(user_query, top_k=3))

print("\n--- 4.أ نتائج التمثيل الهجين التسلسلي (Serial Hybrid) ---")
print(retrieval.search_hybrid_serial(user_query, query_tokens, top_k=3))

print("\n--- 4.ب نتائج التمثيل الهجين التفرعي (Parallel Hybrid) ---")
print(retrieval.search_hybrid_parallel(user_query, query_tokens, alpha=0.5, top_k=3))
print("="*50)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
C:\ProgramData\anaconda3\envs\ir_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. جاري تحميل مجموعة البيانات التجريبية...
2. جاري بناء الفهرس المقلوب الإحصائي...
[CACHE LOADED] cache\cranfield_index.pkl
Loaded index cache: cranfield_index.pkl
3. تهيئة خدمة البحث الشاملة (الطلب الثاني)...
جاري تحميل نموذج BERT (Sentence-Transformer)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4681.04it/s]



4. جاري توليد الـ Embeddings للوثائق باستخدام BERT...
جاري توليد الـ Embeddings للوثائق...


Batches: 100%|██████████| 44/44 [00:04<00:00, 10.19it/s]

تم توليد الـ Embeddings بنجاح!

الاستعلام النصي الأصلي: 'aerodynamic heating structural behavior'
الاستعلام بعد المعالجة (Tokens): ['aerodynam', 'heat', 'structur', 'behavior']

--- 1. نتائج تمثيل VSM (TF-IDF) ---
[('51', 0.7669910830430104), ('142', 0.6844682880849782), ('860', 0.6149609681694912)]

--- 2. نتائج تمثيل BM25 ---
[('51', 8.590784140749406), ('142', 7.43818530743068), ('860', 6.7970216471138105)]

--- 3. نتائج تمثيل الـ Embeddings (BERT) ---
[('142', 0.8496984243392944), ('51', 0.7940446734428406), ('29', 0.7765504121780396)]

--- 4.أ نتائج التمثيل الهجين التسلسلي (Serial Hybrid) ---
[('142', 0.8496984243392944), ('51', 0.7940446734428406), ('29', 0.7765504121780396)]

--- 4.ب نتائج التمثيل الهجين التفرعي (Parallel Hybrid) ---
[('51', 0.8970223367214203), ('142', 0.8577657646964705), ('860', 0.7053648855544324)]


In [1]:

!pip install google-genai


import ir_datasets
from services.preprocessing_service import preprocess
from services.indexing_service import build_inverted_index
from services.retrieval_service import RetrievalService
from services.rag_service import RagService


mini_dataset = ir_datasets.load("cranfield")
total_docs = mini_dataset.docs_count()
index, doc_lengths = build_inverted_index(mini_dataset, preprocess, "cranfield_index.pkl")

retrieval = RetrievalService(index, doc_lengths, total_docs)
retrieval.compute_documents_embeddings(mini_dataset)


rag = RagService(retrieval, mini_dataset)

question = "What is the structural behavior of aerodynamic heating?"
print(f"\n[سؤال المستخدم]: {question}")

rag_response = rag.ask_rag(question, top_k=3)


print("\n" + "="*50)
print("--- إجابة نظام الـ RAG الذكية الفعّالة ---")
print(rag_response["answer"])
print(f"\n[الوثائق الأكاديمية المرجعية التي اعتمد عليها النظام]: {rag_response['referenced_docs']}")
print("="*50)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
C:\ProgramData\anaconda3\envs\ir_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[CACHE LOADED] cache\cranfield_index.pkl
Loaded index cache: cranfield_index.pkl
جاري تحميل نموذج BERT (Sentence-Transformer)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1288.27it/s]


جاري توليد الـ Embeddings للوثائق...


Batches: 100%|██████████| 44/44 [00:17<00:00,  2.52it/s]


تم توليد الـ Embeddings بنجاح!
جاري تهيئة قواميس النصوص لخدمة الـ RAG...

[سؤال المستخدم]: What is the structural behavior of aerodynamic heating?

[RAG] 1. جاري استرجاع أفضل 3 وثائق مرتبطة بالسؤال عبر BERT...
[RAG] 2. جاري دمج النصوص المسترجعة لبناء السياق الحاكم...
[RAG] 3. جاري إرسال البيانات للنموذج اللغوي لتوليد الإجابة النهائية...

--- إجابة نظام الـ RAG الذكية الفعّالة ---
The structural behavior of aerodynamic heating includes transient temperature and thermal stress distribution.

[الوثائق الأكاديمية المرجعية التي اعتمد عليها النظام]: ['142', '51', '29']
